# German (KidSTaLC) WER and error analysis

This notebook evaluates child models on the German KidSTaLC validation set and compares Dense vs MoE behavior with per-utterance and per-word breakdowns. Follow the cells in order.

In [ ]:
# Minimal imports and paths
import json, re, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import matplotlib.pyplot as plt

CHILD_VAL_MANIFEST = Path('/lp-dev/amelia/data/kidstalc/cleaned/val.json')

# Fill this with the correct adult test set path once confirmed.
ADULT_TEST_MANIFEST_CANDIDATES = [
    Path('/data/cv/nemo/de/test.json'),
    Path('/lp-dev/amelia/data/kidstalc/cleaned/test.json'),
    Path('/lp-dev/amelia/data/kidstalc/cleaned/adult_test.json'),
    Path('/lp-dev/amelia/data/kidstalc/cleaned/adult_val.json'),
]

OUTPUT_DIR = Path('analysis_outputs_de')
OUTPUT_DIR.mkdir(exist_ok=True)

print('CHILD_VAL_MANIFEST:', CHILD_VAL_MANIFEST)
print('OUTPUT_DIR:', OUTPUT_DIR)

CHILD_VAL_MANIFEST: /lp-dev/amelia/data/kidstalc/cleaned/val.json
OUTPUT_DIR: analysis_outputs_de


In [2]:
# Step 0 - Data inspection
def load_manifest(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    with open(path, 'r', encoding='utf-8') as f:
        records = [json.loads(l) for l in f if l.strip()]
    return records

def find_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

records = load_manifest(CHILD_VAL_MANIFEST)
print('Child val records:', len(records))
print('Keys:', sorted(records[0].keys()) if records else 'none')
for r in records[:3]:
    print({k: r.get(k) for k in ['audio_filepath', 'text', 'duration']})

adult_manifest = find_first_existing(ADULT_TEST_MANIFEST_CANDIDATES)
print('Adult test manifest:', adult_manifest)
if adult_manifest is None:
    print('Set ADULT_TEST_MANIFEST_CANDIDATES to the correct adult test set path.')

Child val records: 1546
Keys: ['audio_filepath', 'duration', 'text']
{'audio_filepath': '/data/KidsTALC/segments/K003_1-K3-0.wav', 'text': 'nö', 'duration': 0.8}
{'audio_filepath': '/data/KidsTALC/segments/K003_1-K3-1.wav', 'text': 'eh sind ist da überhaupt schrift', 'duration': 3.51}
{'audio_filepath': '/data/KidsTALC/segments/K003_1-K3-2.wav', 'text': 'da kommt einer raus', 'duration': 6.35}
Adult test manifest: None
Set ADULT_TEST_MANIFEST_CANDIDATES to the correct adult test set path.


In [3]:
# Helpers: normalization, alignment, and speaker id
def normalize_german_text(text: str) -> str:
    if not text:
        return ''
    t = text.lower().strip()
    # Normalize unicode then replace German diacritics with ascii-friendly forms
    t = unicodedata.normalize('NFKC', t)
    t = t.replace('ß', 'ss')
    t = t.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue')
    # Keep letters, numbers, and spaces only
    t = re.sub(r"[^a-z0-9\s]", ' ', t)
    t = re.sub(r"\s+", ' ', t).strip()
    return t

def speaker_from_path(audio_path: str) -> str:
    parts = Path(audio_path).parts
    # Adjust if your dataset uses a different layout
    if len(parts) >= 2:
        return parts[-2]
    return 'unknown'

def alignment_counts(ref_words, hyp_words):
    n = len(ref_words)
    m = len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    i, j = n, m
    S = D = I = 0
    ops = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref_words[i - 1] == hyp_words[j - 1]:
            ops.append(('equal', ref_words[i - 1], hyp_words[j - 1]))
            i -= 1; j -= 1
        else:
            if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
                S += 1
                ops.append(('sub', ref_words[i - 1], hyp_words[j - 1]))
                i -= 1; j -= 1
            elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
                D += 1
                ops.append(('del', ref_words[i - 1], None))
                i -= 1
            else:
                I += 1
                ops.append(('ins', None, hyp_words[j - 1]))
                j -= 1
    ops.reverse()
    return S, D, I, ops

def compute_wer(S, D, I, n_words):
    return (S + D + I) / n_words if n_words > 0 else float('inf')

In [4]:
# Step 1 - Model list (edit as needed)
WEIGHTS_DIR = Path('/lp-dev/amelia/inclusive-asr-moe2/final_weights')
MODELS = {
    'dense_child': WEIGHTS_DIR / 'multilingual_child_fastconformer.nemo',
    'moe_child_lb_off': WEIGHTS_DIR / 'multilingual_child_moe_lb_off.nemo',
    'moe_child_lb_on': WEIGHTS_DIR / 'multilingual_child_moe_lb_on.nemo',
    'dense_adult': WEIGHTS_DIR / 'multilingual_adult_fastconformer.nemo',
    'moe_adult': WEIGHTS_DIR / 'multilingual_adult_moe.nemo',
}

for name, path in MODELS.items():
    print(f'{name}: {path} (exists={path.exists()})')

dense_child: /lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_child_fastconformer.nemo (exists=True)
moe_child_lb_off: /lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_child_moe_lb_off.nemo (exists=True)
moe_child_lb_on: /lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_child_moe_lb_on.nemo (exists=True)
dense_adult: /lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_adult_fastconformer.nemo (exists=True)
moe_adult: /lp-dev/amelia/inclusive-asr-moe2/final_weights/multilingual_adult_moe.nemo (exists=True)


In [5]:
# Step 1 - Evaluation loop (child set)
import tarfile
import torch
from nemo.collections.asr.models import EncDecCTCModelBPE

def evaluate_manifest(manifest_path, model_paths, tag):
    records = load_manifest(manifest_path)
    audio_files = [r['audio_filepath'] for r in records]
    refs = [r.get('text', '') for r in records]

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Using device:', device)

    results = {}
    for name, mpath in model_paths.items():
        mpath = Path(mpath)
        print(f'=== {tag}: {name} ===')
        if not mpath.exists():
            results[name] = {'error': 'file not found'}
            print('  missing file, skipping')
            continue
        try:
            tf = tarfile.open(mpath, 'r')
            tf.getmembers()[:1]
            tf.close()
        except Exception as e:
            print('  tar check failed:', e)
        try:
            model = EncDecCTCModelBPE.restore_from(str(mpath), map_location=device)
            model.to(device)
        except Exception as e:
            results[name] = {'error': str(e)}
            print('  restore error:', e)
            continue

        utterances = []
        per_speaker = defaultdict(lambda: {'S':0,'D':0,'I':0,'words':0,'utt':0})
        word_sub = Counter(); word_del = Counter(); word_ins = Counter()
        sub_conf = defaultdict(Counter)

        for audio, ref in zip(audio_files, refs):
            try:
                raw = model.transcribe([audio], batch_size=1, num_workers=0, verbose=False)
                hyp = raw[0].text if hasattr(raw[0], 'text') else str(raw[0])
            except Exception as e:
                hyp = f'__ERROR__:{e}'
            ref_n = normalize_german_text(ref)
            hyp_n = normalize_german_text(hyp)
            ref_w = ref_n.split()
            hyp_w = hyp_n.split()
            S, D, I, ops = alignment_counts(ref_w, hyp_w)
            sp = speaker_from_path(audio)
            per_speaker[sp]['S'] += S
            per_speaker[sp]['D'] += D
            per_speaker[sp]['I'] += I
            per_speaker[sp]['words'] += len(ref_w)
            per_speaker[sp]['utt'] += 1
            for op, rw, hw in ops:
                if op == 'del' and rw:
                    word_del[rw] += 1
                elif op == 'ins' and hw:
                    word_ins[hw] += 1
                elif op == 'sub' and rw is not None and hw is not None:
                    word_sub[rw] += 1
                    sub_conf[rw][hw] += 1
            utterances.append({
                'audio': audio,
                'speaker': sp,
                'ref': ref_n,
                'hyp': hyp_n,
                'S': S,
                'D': D,
                'I': I,
                'ref_words': len(ref_w),
                'wer': compute_wer(S, D, I, len(ref_w))
            })

        speaker_rows = []
        total_S = total_D = total_I = total_words = 0
        for sp, v in per_speaker.items():
            S=v['S']; D=v['D']; I=v['I']; W=v['words']
            speaker_rows.append({
                'speaker': sp, 'S': S, 'D': D, 'I': I, 'words': W, 'utt': v['utt'],
                'WER': compute_wer(S, D, I, W)
            })
            total_S+=S; total_D+=D; total_I+=I; total_words+=W

        summary = {
            'S': total_S, 'D': total_D, 'I': total_I, 'words': total_words,
            'WER': compute_wer(total_S, total_D, total_I, total_words)
        }
        results[name] = {
            'utterances': utterances,
            'per_speaker': pd.DataFrame(sorted(speaker_rows, key=lambda r: r['WER'], reverse=True)),
            'word_del_counts': word_del,
            'word_sub_counts': word_sub,
            'word_ins_counts': word_ins,
            'substitution_confusions': {k: dict(v) for k, v in sub_conf.items()},
            'summary': summary
        }
        print('  done, WER:', summary['WER'])
    return results

child_results = evaluate_manifest(CHILD_VAL_MANIFEST, MODELS, tag='child')

# Save per-model summary for quick inspection
summary_rows = []
for name, r in child_results.items():
    if isinstance(r, dict) and 'error' in r:
        summary_rows.append({'model': name, 'error': r['error']})
        continue
    s = r['summary']
    summary_rows.append({'model': name, **s})
df_child_summary = pd.DataFrame(summary_rows)
df_child_summary.to_csv(OUTPUT_DIR / 'child_model_summary.csv', index=False)
print(df_child_summary)

/home/nvidia/miniconda3/envs/nemo_asr/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


OSError: Could not load this library: /home/nvidia/miniconda3/envs/nemo_asr/lib/python3.12/site-packages/torchaudio/lib/_torchaudio.abi3.so

In [ ]:
# Step 2 - Overall S/D/I breakdown per model
if 'child_results' not in globals() or not child_results:
    print('Run the evaluation cell first.')
else:
    rows = []
    for name, r in child_results.items():
        if isinstance(r, dict) and 'error' in r:
            rows.append({'model': name, 'error': r['error']})
            continue
        s = r['summary']
        rows.append({'model': name, 'S': s['S'], 'D': s['D'], 'I': s['I'], 'words': s['words'], 'WER': s['WER']})
    df_models = pd.DataFrame(rows).set_index('model')
    df_models.to_csv(OUTPUT_DIR / 'child_SDI_breakdown.csv')
    print(df_models)

    # Plot WER per model
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_df = df_models.dropna(subset=['WER'])
    if not plot_df.empty:
        plot_df['WER'].sort_values().plot(kind='bar', ax=ax)
        ax.set_ylabel('WER')
        ax.set_title('KidSTaLC child WER by model')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        fig.savefig(OUTPUT_DIR / 'child_WER_by_model.png')
        plt.close(fig)

In [ ]:
# Step 3 - Head-to-head Dense vs MoE (child models)
if 'child_results' not in globals() or not child_results:
    print('Run the evaluation cell first.')
else:
    base = child_results.get('dense_child')
    moe_off = child_results.get('moe_child_lb_off')
    moe_on = child_results.get('moe_child_lb_on')
    if not base or not moe_off or not moe_on:
        print('Missing one or more child model results.')
    else:
        # Map by audio path for consistent comparison
        def to_map(results):
            return {u['audio']: u for u in results['utterances']}
        dense_map = to_map(base)
        moe_off_map = to_map(moe_off)
        moe_on_map = to_map(moe_on)

        rows = []
        for audio, d in dense_map.items():
            m_off = moe_off_map.get(audio)
            m_on = moe_on_map.get(audio)
            if not m_off or not m_on:
                continue
            rows.append({
                'audio': audio,
                'speaker': d['speaker'],
                'ref_words': d['ref_words'],
                'wer_dense': d['wer'],
                'wer_moe_off': m_off['wer'],
                'wer_moe_on': m_on['wer'],
            })
        df_cmp = pd.DataFrame(rows)
        df_cmp.to_csv(OUTPUT_DIR / 'dense_vs_moe_utterance.csv', index=False)
        print('Rows:', len(df_cmp))

        def bucket(row):
            best = min(row['wer_dense'], row['wer_moe_off'], row['wer_moe_on'])
            if row['wer_dense'] == best and row['wer_moe_off'] != best and row['wer_moe_on'] != best:
                return 'dense_wins'
            if row['wer_moe_off'] == best or row['wer_moe_on'] == best:
                if row['wer_dense'] == best:
                    return 'tie_dense_moe'
                return 'moe_wins'
            return 'tie'

        df_cmp['bucket'] = df_cmp.apply(bucket, axis=1)
        print(df_cmp['bucket'].value_counts())
        df_cmp.groupby('bucket')[['ref_words','wer_dense','wer_moe_off','wer_moe_on']].mean()

In [ ]:
# Step 4 - Per-word error analysis (full sentence)
if 'child_results' not in globals() or not child_results:
    print('Run the evaluation cell first.')
else:
    for name, r in child_results.items():
        if isinstance(r, dict) and 'error' in r:
            print(name, 'error:', r['error'])
            continue
        word_sub = r['word_sub_counts']
        word_del = r['word_del_counts']
        word_ins = r['word_ins_counts']
        sub_conf = r['substitution_confusions']

        # Top words by errors
        top_sub = word_sub.most_common(20)
        top_del = word_del.most_common(20)
        top_ins = word_ins.most_common(20)
        pd.DataFrame(top_sub, columns=['ref_word', 'count']).to_csv(OUTPUT_DIR / f'{name}_top_sub_words.csv', index=False)
        pd.DataFrame(top_del, columns=['ref_word', 'count']).to_csv(OUTPUT_DIR / f'{name}_top_del_words.csv', index=False)
        pd.DataFrame(top_ins, columns=['hyp_word', 'count']).to_csv(OUTPUT_DIR / f'{name}_top_ins_words.csv', index=False)

        # Top substitution pairs
        pairs = []
        for ref_word, hyp_counts in sub_conf.items():
            for hyp_word, cnt in hyp_counts.items():
                pairs.append({'ref_word': ref_word, 'hyp_word': hyp_word, 'count': cnt})
        df_pairs = pd.DataFrame(pairs).sort_values('count', ascending=False).head(50)
        df_pairs.to_csv(OUTPUT_DIR / f'{name}_top_sub_pairs.csv', index=False)
        print('Saved word error files for', name)

In [ ]:
# Step 5 - Per-speaker breakdown
if 'child_results' not in globals() or not child_results:
    print('Run the evaluation cell first.')
else:
    for name, r in child_results.items():
        if isinstance(r, dict) and 'error' in r:
            continue
        df_sp = r['per_speaker']
        out_csv = OUTPUT_DIR / f'{name}_per_speaker.csv'
        df_sp.to_csv(out_csv, index=False)
        # Plot top speakers by WER
        top_n = min(40, len(df_sp))
        if top_n == 0:
            continue
        df_top = df_sp.head(top_n)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(df_top['speaker'].astype(str), df_top['WER'] * 100)
        ax.set_xlabel('WER (%)')
        ax.set_title(f'Top {top_n} speakers by WER - {name}')
        ax.invert_yaxis()
        plt.tight_layout()
        fig.savefig(OUTPUT_DIR / f'{name}_top_speakers.png')
        plt.close(fig)
        print('Saved per-speaker outputs for', name)

In [ ]:
# Step 6 - WER vs utterance length
if 'child_results' not in globals() or not child_results:
    print('Run the evaluation cell first.')
else:
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, r in child_results.items():
        if isinstance(r, dict) and 'error' in r:
            continue
        df_u = pd.DataFrame(r['utterances'])
        if df_u.empty:
            continue
        ax.scatter(df_u['ref_words'], df_u['wer'], alpha=0.35, label=name, s=12)
    ax.set_xlabel('Reference words')
    ax.set_ylabel('Per-utterance WER')
    ax.set_title('WER vs utterance length (child set)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'wer_vs_length_child.png')
    plt.close(fig)

In [ ]:
# Step 7 - Adult degradation analysis (child models on adult set)
if adult_manifest is None:
    print('Adult manifest not set. Update ADULT_TEST_MANIFEST_CANDIDATES and re-run Step 0.')
else:
    # Use only child-trained models for adult degradation analysis
    child_only = {
        'dense_child': MODELS['dense_child'],
        'moe_child_lb_off': MODELS['moe_child_lb_off'],
        'moe_child_lb_on': MODELS['moe_child_lb_on'],
    }
    adult_results = evaluate_manifest(adult_manifest, child_only, tag='adult')
    rows = []
    for name, r in adult_results.items():
        if isinstance(r, dict) and 'error' in r:
            rows.append({'model': name, 'error': r['error']})
            continue
        s = r['summary']
        rows.append({'model': name, **s})
    df_adult_summary = pd.DataFrame(rows)
    df_adult_summary.to_csv(OUTPUT_DIR / 'adult_model_summary.csv', index=False)
    print(df_adult_summary)

In [ ]:
# Step 8 - Summary outputs
if 'child_results' in globals() and child_results:
    summary_rows = []
    for name, r in child_results.items():
        if isinstance(r, dict) and 'error' in r:
            summary_rows.append({'model': name, 'set': 'child', 'error': r['error']})
            continue
        s = r['summary']
        summary_rows.append({'model': name, 'set': 'child', **s})
    if 'adult_results' in globals():
        for name, r in adult_results.items():
            if isinstance(r, dict) and 'error' in r:
                summary_rows.append({'model': name, 'set': 'adult', 'error': r['error']})
                continue
            s = r['summary']
            summary_rows.append({'model': name, 'set': 'adult', **s})
    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(OUTPUT_DIR / 'combined_summary.csv', index=False)
    print(df_summary)
else:
    print('Run evaluation first.')